# Session 07 Practice: Reactive Planning & Trajectory Rollouts
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 4 | Session 7**  

---

### 🎯 Learning Objectives
1. Generate candidate **trajectory rollout bundles** via forward kinematic bicycle propagation.
2. Implement **multi-circle bounding approximation** for vehicle footprint collision checking.
3. Formulate multi-objective rollout scoring: $J = w_{\text{lane}} |y_{\text{end}} - y_{\text{goal}}| + w_{\text{steer}} |\delta|$.
4. Filter colliding paths and select the optimal reactive path using **Plotly**.

---

### 📐 Mathematical Formulation

#### Trajectory Generation
$$\dot{x} = v \cos\theta, \quad \dot{y} = v \sin\theta, \quad \dot{\theta} = \frac{v}{L} \tan\delta_i$$

#### Multi-Circle Footprint Geometry
A vehicle with bounding radius $R$ is modeled by circles centered along its centerline:
$$c_{k, x} = x_k + d_m \cos\theta_k, \quad c_{k, y} = y_k + d_m \sin\theta_k, \quad d_m \in \{-1.0, 0.5, 2.0\}$$
Collision with obstacle at $(o_x, o_y, r_{\text{obs}})$ occurs if:
$$\sqrt{(c_{k, x} - o_x)^2 + (c_{k, y} - o_y)^2} < R_{\text{circle}} + r_{\text{obs}}$$


In [ ]:
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. Multi-Circle Collision Checker & Rollout Generator

In [ ]:
class CircleCollisionChecker:
    def __init__(self, circle_offsets: list[float] = [-1.0, 0.5, 2.0], circle_radius: float = 1.25):
        self.offsets = circle_offsets
        self.radius = circle_radius

    def is_collision(self, traj_points: list[tuple], obstacles: list[tuple]) -> bool:
        for x, y, theta in traj_points:
            for d in self.offsets:
                cx = x + d * math.cos(theta)
                cy = y + d * math.sin(theta)
                for ox, oy, r_obs in obstacles:
                    dist = math.hypot(cx - ox, cy - oy)
                    if dist < (self.radius + r_obs):
                        return True
        return False

def generate_rollouts(x0: float = 0.0, y0: float = 0.0, theta0: float = 0.0, v0: float = 8.0,
                      num_paths: int = 21, max_steer: float = 0.35,
                      horizon: float = 3.0, dt: float = 0.1, L: float = 2.8) -> list[dict]:
    rollouts = []
    steer_angles = np.linspace(-max_steer, max_steer, num_paths)
    num_steps = int(horizon / dt)
    
    for idx, delta in enumerate(steer_angles):
        path = []
        x, y, theta, v = x0, y0, theta0, v0
        for _ in range(num_steps):
            x += v * math.cos(theta) * dt
            y += v * math.sin(theta) * dt
            theta += (v / L) * math.tan(delta) * dt
            path.append((x, y, theta))
            
        rollouts.append({
            "rollout_id": idx,
            "steer_angle_rad": delta,
            "path": path,
            "end_x": path[-1][0],
            "end_y": path[-1][1],
        })
    return rollouts

def score_and_select_best(rollouts: list[dict], obstacles: list[tuple], goal_y: float = 0.0):
    checker = CircleCollisionChecker()
    scored = []
    best_rollout = None
    min_cost = float('inf')
    
    for r in rollouts:
        colliding = checker.is_collision(r["path"], obstacles)
        if colliding:
            cost = float('inf')
        else:
            cost = abs(r["end_y"] - goal_y) + 0.8 * abs(r["steer_angle_rad"])
            if cost < min_cost:
                min_cost = cost
                best_rollout = r
                
        scored.append({
            "rollout_id": r["rollout_id"],
            "steer_angle_rad": round(r["steer_angle_rad"], 3),
            "is_colliding": colliding,
            "cost": round(cost, 3) if cost != float('inf') else 999.0,
            "end_x": round(r["end_x"], 2),
            "end_y": round(r["end_y"], 2)
        })
        
    return best_rollout, pl.DataFrame(scored)


## 2. Rollout Evaluation with Static Obstacle

In [ ]:
# Obstacle directly ahead at (15m, 0m)
obstacles = [(15.0, 0.0, 1.5)]

rollouts = generate_rollouts(x0=0.0, y0=0.0, theta0=0.0, v0=8.0, num_paths=21, max_steer=0.35)
best_rollout, df_scored = score_and_select_best(rollouts, obstacles, goal_y=0.0)

print(f"Selected Best Rollout ID: {best_rollout['rollout_id']} with steer={best_rollout['steer_angle_rad']:.3f} rad")


## 3. Interactive Rollout Visualizer & Cost Profile Plots

In [ ]:
# 1. 2D Spatial Rollout Bundle Plot
fig_bundle = go.Figure()

# Obstacles
for ox, oy, r_obs in obstacles:
    fig_bundle.add_shape(type="circle",
                         xref="x", yref="y",
                         x0=ox - r_obs, y0=oy - r_obs,
                         x1=ox + r_obs, y1=oy + r_obs,
                         fillcolor="crimson", line_color="darkred", opacity=0.8)

# Rollouts
for r in rollouts:
    r_id = r["rollout_id"]
    is_col = df_scored.filter(pl.col("rollout_id") == r_id)["is_colliding"][0]
    color = "rgba(231, 76, 60, 0.35)" if is_col else "rgba(52, 152, 219, 0.5)"
    xs = [p[0] for p in r["path"]]
    ys = [p[1] for p in r["path"]]
    fig_bundle.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines",
        line=dict(color=color, width=2),
        showlegend=False,
        hoverinfo="text",
        text=f"Rollout {r_id} (Col: {is_col})"
    ))

# Best Rollout
if best_rollout:
    b_xs = [p[0] for p in best_rollout["path"]]
    b_ys = [p[1] for p in best_rollout["path"]]
    fig_bundle.add_trace(go.Scatter(
        x=b_xs, y=b_ys, mode="lines+markers",
        line=dict(color="#2ecc71", width=4.5),
        marker=dict(size=4),
        name=f"Optimal Rollout #{best_rollout['rollout_id']}"
    ))

fig_bundle.update_layout(
    title="Reactive Trajectory Rollouts with Multi-Circle Obstacle Avoidance",
    xaxis_title="Global X (m)",
    yaxis_title="Global Y (m)",
    width=850,
    height=600
)
fig_bundle.show()


In [ ]:
# 2. Rollout Objective Cost Profile
fig_cost = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Candidate Trajectory Cost Function", "Candidate Final Lateral Offsets (Y_end)")
)

valid_mask = df_scored["is_colliding"] == False
colors = ["#2ecc71" if not col else "#e74c3c" for col in df_scored["is_colliding"]]

fig_cost.add_trace(
    go.Bar(x=df_scored["steer_angle_rad"], y=df_scored["cost"], marker_color=colors, name="Objective Cost"),
    row=1, col=1
)
fig_cost.add_trace(
    go.Scatter(x=df_scored["steer_angle_rad"], y=df_scored["end_y"], mode="lines+markers",
               line=dict(color="#1f77b4", width=2.5), name="End Y (m)"),
    row=1, col=2
)
fig_cost.add_hline(y=0.0, line_dash="dash", line_color="gray", annotation_text="Goal Centerline", row=1, col=2)

fig_cost.update_layout(title="Trajectory Selection Cost Analysis", height=420, showlegend=False)
fig_cost.show()


## 4. Verification Tests

In [ ]:
# Verify that chosen rollout avoids the obstacle
checker = CircleCollisionChecker()
assert not checker.is_collision(best_rollout["path"], obstacles), "Optimal path must be strictly collision-free!"
assert abs(best_rollout["steer_angle_rad"]) > 0.05, "Vehicle must steer away from centerline obstacle!"

print("✅ Trajectory rollout tests passed successfully!")
